In [1]:
# Environment Setup & Authentication
# from google.colab import userdata
from huggingface_hub import login
import warnings

# local environment
import os
# from dotenv import load_dotenv

# Load variables from .env
# load_dotenv()

# Suppress the specific warning about missing tokens
warnings.filterwarnings(
    "ignore", category=UserWarning, module="huggingface_hub.utils._auth"
)

try:
    # Attempt to fetch the token from Colab's secure storage
    # hf_token = userdata.get("HF_TOKEN")

    # Get the token from the local environment
    hf_token = os.getenv("HF_TOKEN")

    # Log into the Hugging Face Hub using the retrieved token
    login(hf_token)
    
    print("Authentication successful.")

# except userdata.SecretNotFoundError:
#     # If the token isn't found, smoothly continue without crashing
#     print(
#         "Notice: HF_TOKEN not found in Colab secrets. Proceeding without authentication."
#     )
except Exception as error:
    # Handle any other unexpected exceptions
    print(f"An unexpected error occurred during authentication: {error}")

Authentication successful.


In [2]:
from pandas import read_csv, DataFrame, concat
from typing import Any

df = read_csv("../../data/prepared/laboratory_data_clean.csv")

leakage_cols = [
    "dialysis",
    "insulin_duration_unit",
    "insulin_duration",
    "age_diabetes_diagnosed",
    "taking_insulin",
    "taking_diabetes_pills",
    "diabetes_blood_test_recently",
    "prediabetes",
]

df = df.drop(columns=leakage_cols)

ALL_TARGET_COLUMNS = [
    "diabetes",
    "kidney_disease",
    "cancer",
    "stroke",
    "liver_condition",
    "chronic_obstructive_pulmonary_disease",
    "coronary_heart_disease",
    "thyroid_condition",
]

DISEASE_COLUMNS = {
    "diabetes": "diabetes",
    "kidney_disease": "kidney_disease",
    "cancer": "cancer",
    # "stroke": "stroke",
    # "liver_condition": "liver_condition",
    # "copd": "chronic_obstructive_pulmonary_disease",
    # "heart_disease": "coronary_heart_disease",
    # "thyroid_condition": "thyroid_condition",
}

FEATURE_COLUMNS = [column for column in df.columns if column not in ALL_TARGET_COLUMNS]

required_columns = FEATURE_COLUMNS + list(DISEASE_COLUMNS.values())


def balance_dataset(
    disease_df: DataFrame,
    target_column: str,
    max_samples: int = 900,
    positive_ratio: float = 0.3,
) -> DataFrame:

    positive_df = disease_df[disease_df[target_column] == 1]
    negative_df = disease_df[disease_df[target_column] == 0]

    positive_count = min(
        len(positive_df),
        int(max_samples * positive_ratio),
    )

    negative_count = min(
        len(negative_df),
        max_samples - positive_count,
    )

    positive_sample = positive_df.sample(
        n=positive_count,
        random_state=42,
    )

    negative_sample = negative_df.sample(
        n=negative_count,
        random_state=42,
    )

    balanced_df = concat([positive_sample, negative_sample])

    return balanced_df.sample(
        frac=1,
        random_state=42,
    ).reset_index(drop=True)


def disease_dataset(disease):
    target_column = DISEASE_COLUMNS[disease]

    required_columns = [*FEATURE_COLUMNS, target_column] # spread the feature columns and add the target column to the list

    df_model = df[required_columns].dropna().copy()

    # balance the dataset to have a maximum of 100 samples with a positive ratio of 0.3 so training is not biased towards the negative class and can be done quickly
    
    # df_model = balance_dataset(
    #     disease_df=df_model,
    #     target_column=target_column,
    #     max_samples=100,
    #     positive_ratio=0.3,
    # )

    print("\nDataFrame columns:")
    print(df_model.columns.tolist())

    print("\nClass distribution:")
    print(df_model[disease].value_counts(normalize=True))

    return df_model


disease_datasets = {
    disease: disease_dataset(disease) for disease in DISEASE_COLUMNS.keys()
}


# format float values to 3 decimal places for better readability and to save tokens when fine-tuning
def format_value(value):
    if isinstance(value, float):
        return round(value, 3)

    return value


# create a prompt for disease prediction based on the patient data
def create_prompt(data_row: Any, disease: str) -> str:
    lines = [
        f"Determine whether the patient has {disease}:"
    ]

    for column in FEATURE_COLUMNS:
        value = format_value(getattr(data_row, column))

        lab_data_title = column.title()

        lines.append(f"{lab_data_title}: {value}")

    return "\n".join(lines)


DataFrame columns:
['age', 'gender', 'ethnicity', 'glucose_mg_dl', 'hba1c_percent', 'hemoglobin_g_dl', 'hematocrit_percent', 'wbc_10_3_per_ul', 'platelets_10_3_per_ul', 'creatinine_mg_dl', 'bun_mg_dl', 'sodium_mmol_l', 'potassium_mmol_l', 'chloride_mmol_l', 'calcium_mg_dl', 'albumin_g_dl', 'total_protein_g_dl', 'alt_u_l', 'ast_u_l', 'bilirubin_mg_dl', 'total_cholesterol_mg_dl', 'hdl_mg_dl', 'triglycerides_mg_dl', 'diabetes']

Class distribution:
diabetes
0.0    0.905606
1.0    0.094394
Name: proportion, dtype: float64

DataFrame columns:
['age', 'gender', 'ethnicity', 'glucose_mg_dl', 'hba1c_percent', 'hemoglobin_g_dl', 'hematocrit_percent', 'wbc_10_3_per_ul', 'platelets_10_3_per_ul', 'creatinine_mg_dl', 'bun_mg_dl', 'sodium_mmol_l', 'potassium_mmol_l', 'chloride_mmol_l', 'calcium_mg_dl', 'albumin_g_dl', 'total_protein_g_dl', 'alt_u_l', 'ast_u_l', 'bilirubin_mg_dl', 'total_cholesterol_mg_dl', 'hdl_mg_dl', 'triglycerides_mg_dl', 'kidney_disease']

Class distribution:
kidney_disease
0.0

In [3]:
from pathlib import Path
import json

output_dir = Path("../../data/fine_tuning")
output_dir.mkdir(parents=True, exist_ok=True)

for disease, disease_df in disease_datasets.items():

    output_file = output_dir / f"{disease}.jsonl"

    with output_file.open("w", encoding="utf-8") as file:

        target_column = DISEASE_COLUMNS[disease]

        for row in disease_df.itertuples(index=False):

            prompt = create_prompt(row, disease)
            
            # Get the target value for the current row using getattr (the disease value either 0 or 1)
            target_value = getattr(row, target_column)

            # Determine the prediction based on the target column value positive if the value is 1, otherwise negative
            prediction = "positive" if target_value == 1 else "negative"

            training_data = {
                "prompt": prompt,
                "completion": prediction,
            }

            file.write(json.dumps(training_data, ensure_ascii=False) + "\n")

    print(f"Created: {output_file}")

Created: ..\..\data\fine_tuning\diabetes.jsonl
Created: ..\..\data\fine_tuning\kidney_disease.jsonl
Created: ..\..\data\fine_tuning\cancer.jsonl


In [4]:
# Load Model and Tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM,BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# Check if GPU is available and set the device accordingly
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Load model using the appropriate data type based on the device
def load_base_model():
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
        
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map=device,
        quantization_config=quantization_config, # for 4-bit quantization (can be removed if not using 4-bit quantization) used to reduce VRAM usage and speed up inference
    )
    model.config.use_cache = False

    return model

In [11]:
# Baseline Inference (Before Fine-Tuning)
messages = [
    {
        "role": "user",
        "content": """"Patient Demographics: Gender=M. Lab Results: Glucose: 104.6 mg/dL,
    Urea Nitrogen: 12.7 mg/dL, Calcium: 10.1 mg/dL, Sodium: 136.9 mmol/L,
    Potassium: 4.3 mmol/L, Chloride: 107.8 mmol/L, Carbon Dioxide: 20.6 mmol/L, HbA1c: 4.4 %
""",
    }
]

model = load_base_model()

# Format the prompt
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)
# print("Token IDs:\n", inputs["input_ids"][0][:20]) # Shows the first ~20 token IDs (numbers)

# Generate an answer using the untrained starting model
output = model.generate(
    **inputs, max_new_tokens=512, do_sample=False, eos_token_id=tokenizer.eos_token_id
)

# Extract and print only the generated response
generated_ids = output[0][inputs.input_ids.shape[1] :]
print(
    "STARTING MODEL ANSWER:\n",
    tokenizer.decode(generated_ids, skip_special_tokens=True),
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

STARTING MODEL ANSWER:
 Based on the provided patient demographics and lab results, here is an analysis:

**Demographics:**
- **Gender:** Male

**Lab Results:**
- **Glucose:** 104.6 mg/dL (Normal range for men is typically between 70-150 mg/dL)
- **Urea Nitrogen:** 12.7 mg/dL (Normal range varies but generally less than 7.5 mg/dL in healthy individuals)
- **Calcium:** 10.1 mg/dL (Normal range is usually between 8.5-10.5 mg/dL)
- **Sodium:** 136.9 mmol/L (Normal range is typically between 130-145 mmol/L)
- **Potassium:** 4.3 mmol/L (Normal range is usually between 3.5-5.5 mmol/L)
- **Chloride:** 107.8 mmol/L (Normal range is typically between 98-108 mmol/L)
- **Carbon Dioxide:** 20.6 mmol/L (Normal range is typically between 22-32 mmol/L)
- **HbA1c:** 4.4% (Normal range is typically below 5%)

The glucose level of 104.6 mg/dL is slightly elevated, which could indicate prediabetes or diabetes if it persists over time. The other lab values do not suggest any significant health issues base

In [6]:
# Dataset Loading and Preprocessing
from datasets import load_dataset

def load_dataset_from_jsonl(filename:str):
  return load_dataset("json", data_files=filename)


def preprocess(sample):
    messages = [
        {"role": "user", "content": sample["prompt"]},
        {"role": "assistant", "content": sample["completion"]},
    ]

    # Automatically applies <|im_start|> and <|im_end|> ChatML tags
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=512, # Set a maximum length for the tokenized input (you can adjust this as needed)
        padding=False,
    )
    
    # Explicitly create labels for loss calculation
    # tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

diabetes_raw_data = load_dataset_from_jsonl("../../data/fine_tuning/diabetes.jsonl")
cancer_raw_data = load_dataset_from_jsonl("../../data/fine_tuning/cancer.jsonl")
kidney_disease_raw_data = load_dataset_from_jsonl("../../data/fine_tuning/kidney_disease.jsonl")
# copd_raw_data = load_dataset_from_jsonl(
#     "../../data/fine_tuning/chronic_obstructive_pulmonary_disease.jsonl"
# )
# liver_condition_raw_data = load_dataset_from_jsonl("../../data/fine_tuning/liver_condition.jsonl")
# stroke_raw_data = load_dataset_from_jsonl("../../data/fine_tuning/stroke.jsonl")
# coronary_heart_disease_raw_data = load_dataset_from_jsonl("../../data/fine_tuning/coronary_heart_disease.jsonl")
# thyroid_condition_raw_data = load_dataset_from_jsonl(
#     "../../data/fine_tuning/thyroid_condition.jsonl"
# )

diabetes_data = diabetes_raw_data.map(
    preprocess,
    remove_columns=diabetes_raw_data["train"].column_names,
)

cancer_data = cancer_raw_data.map(
    preprocess,
    remove_columns=cancer_raw_data["train"].column_names,
)

kidney_disease_data = kidney_disease_raw_data.map(
    preprocess,
    remove_columns=kidney_disease_raw_data["train"].column_names,
)

# copd_data = copd_raw_data.map(
#     preprocess,
#     remove_columns=copd_raw_data["train"].column_names,
# )

# liver_condition_data = liver_condition_raw_data.map(
#     preprocess,
#     remove_columns=liver_condition_raw_data["train"].column_names,
# )

# stroke_data = stroke_raw_data.map(
#     preprocess,
#     remove_columns=stroke_raw_data["train"].column_names,
# )

# coronary_heart_disease_data = coronary_heart_disease_raw_data.map(
#     preprocess,
#     remove_columns=coronary_heart_disease_raw_data["train"].column_names,
# )

# thyroid_condition_data = thyroid_condition_raw_data.map(
#     preprocess,
#     remove_columns=thyroid_condition_raw_data["train"].column_names,
# )

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/11452 [00:00<?, ? examples/s]

Map:   0%|          | 0/7800 [00:00<?, ? examples/s]

Map:   0%|          | 0/7794 [00:00<?, ? examples/s]

In [7]:
# LoRA Configuration
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
)

model = get_peft_model(model, lora_config)

In [8]:
from peft import get_peft_model
from transformers import (
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)
import gc
import torch

def train_disease_adapter(
    base_model,
    dataset,
    output_dir,
):
    # Create a fresh LoRA adapter
    model = get_peft_model(
        base_model,
        lora_config,
    )

    model.print_trainable_parameters()

    # Split 10% for validation
    split = dataset["train"].train_test_split(
        test_size=0.1,
        seed=42,
    )

    train_dataset = split["train"]
    eval_dataset = split["test"]

    # Padding + causal-LM labels
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=5,
        learning_rate=2e-4,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=2,
        fp16=True,
        logging_steps=5,
        eval_strategy="epoch",
        lr_scheduler_type="cosine",
        remove_unused_columns=False,
        optim="paged_adamw_8bit", # for 8-bit optimization to reduce VRAM usage and speed up training
    )

    model.gradient_checkpointing_enable()

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
    )

    trainer.train()

    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    # Release GPU memory
    del trainer
    del model
    del base_model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return eval_dataset, train_dataset

In [9]:
model = load_base_model()

train_disease_adapter(
    base_model=model,
    dataset=diabetes_data,
    output_dir="./adapters/diabetes",
)

model = load_base_model()

train_disease_adapter(
    base_model=model,
    dataset=cancer_data,
    output_dir="./adapters/cancer",
)

# model = load_base_model()

# train_disease_adapter(
#     base_model=model,
#     dataset=kidney_disease_data,
#     output_dir="./adapters/kidney_disease",
# )

# model = load_base_model()

# train_disease_adapter(
#     base_model=model,
#     dataset=liver_condition_data,
#     output_dir="./adapters/liver_condition",
# )

# model = load_base_model()

# train_disease_adapter(
#     base_model=model,
#     dataset=stroke_data,
#     output_dir="./adapters/stroke",
# )

# model = load_base_model()

# train_disease_adapter(
#     base_model=model,
#     dataset=coronary_heart_disease_data,
#     output_dir="./adapters/coronary_heart_disease",
# )


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:

# Zip all disease adapters for colab download
# !zip -r disease_adapters.zip adapters

# Download the ZIP
# from google.colab import files

# files.download("disease_adapters.zip")

# for local testing, you can zip the adapters folder to upload to Hugging Face Hub
import shutil

zip_filename = "../../data/disease_adapters.zip"

shutil.make_archive(
    "disease_adapters",
    "zip",
    "adapters",
)

'd:\\programming\\metropolia\\data_handling\\data_handling_project\\src\\notebooks\\disease_adapters.zip'

In [ ]:
# If you have restarted the Colab runtime, you can re-upload the ZIP file and extract it to continue testing without retraining.
# Load Model for Testing
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM
import torch

# Folder containing your extracted adapters
adapter_root = "./adapters"

# Load the base Qwen model ONCE
config = PeftConfig.from_pretrained(f"{adapter_root}/diabetes")

base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    device_map="cuda",
    dtype=torch.float16,
)

base_model.config.use_cache = True

# Load the first adapter and create the PEFT model
model = PeftModel.from_pretrained(
    base_model,
    f"{adapter_root}/diabetes",
    adapter_name="diabetes",
)

# Load additional adapters into the same model
def load_adapter_for_disease(filepath: str, adapter_name: str):
    # Load the adapter for the specified disease into the model
    try:
        model.load_adapter(
        f"{adapter_root}/{filepath}",
        adapter_name=adapter_name,
    )
    # catch any exceptions that occur during loading and print an error message so that the program continues to run
    except Exception as error:
        print(f"Error loading adapter for {adapter_name}: {error}")

load_adapter_for_disease("cancer", "cancer")
# load_adapter_for_disease("kidney_disease", "kidney_disease")
# load_adapter_for_disease("copd", "copd")
# load_adapter_for_disease("stroke", "stroke")
# load_adapter_for_disease("liver_condition", "liver_condition")
# load_adapter_for_disease("coronary_heart_disease", "coronary_heart_disease")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
# Diabetes Inference
model.set_adapter("cancer")  # Switch to the cancer adapter for inference

messages = [
    {
        "role": "user",
        "content": """Detemine whether the patient has cancer based on the following patient data:

    Age: 80.0
    Gender: Male
    Ethnicity: Non-Hispanic White
    Glucose_Mg_Dl: 100.0
    Hba1C_Percent: 5.5
    Hemoglobin_G_Dl: 13.7
    Hematocrit_Percent: 40.6
    Wbc_10_3_Per_Ul: 6.7
    Platelets_10_3_Per_Ul: 259.0
    Creatinine_Mg_Dl: 0.83
    Bun_Mg_Dl: 14.0
    Sodium_Mmol_L: 140.0
    Potassium_Mmol_L: 4.1
    Chloride_Mmol_L: 102.0
    Calcium_Mg_Dl: 9.4
    Albumin_G_Dl: 4.1
    Total_Protein_G_Dl: 7.2
    Alt_U_L: 17.0
    Ast_U_L: 20.0
    Bilirubin_Mg_Dl: 0.4
    Total_Cholesterol_Mg_Dl: 178.0
    Hdl_Mg_Dl: 52.0
    Triglycerides_Mg_Dl: 96.0
""",
    }
]

# Format the text with ChatML tags and generation prompt
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

# Convert text to tensor numbers and move to GPU
inputs = tokenizer(
    text,
    return_tensors="pt",
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id,
)

# Strip out the input prompt so we only see the newly generated answer
generated_ids = output[0][inputs.input_ids.shape[1] :]
print("FINE-TUNED ANSWER:\n", tokenizer.decode(generated_ids, skip_special_tokens=True))

FINE-TUNED ANSWER:
 negative


In [ ]:
diseases = [
    "diabetes",
    "cancer",
]


def create_content_for_disease_prediction(disease: str):
    return f"""Determine whether the patient has {disease} based on the following patient data:
Age: 61.0
Gender: Male
Ethnicity: Non-Hispanic White
Glucose_Mg_Dl: 88.0
Hba1C_Percent: 5.2
Hemoglobin_G_Dl: 14.0
Hematocrit_Percent: 40.8
Wbc_10_3_Per_Ul: 4.5
Platelets_10_3_Per_Ul: 177.0
Creatinine_Mg_Dl: 1.07
Bun_Mg_Dl: 16.0
Sodium_Mmol_L: 141.0
Potassium_Mmol_L: 4.3
Chloride_Mmol_L: 104.0
Calcium_Mg_Dl: 9.2
Albumin_G_Dl: 4.0
Total_Protein_G_Dl: 6.7
Alt_U_L: 15.0
Ast_U_L: 22.0
Bilirubin_Mg_Dl: 0.6
Total_Cholesterol_Mg_Dl: 189.0
Hdl_Mg_Dl: 49.0
Triglycerides_Mg_Dl: 64.0
    """


results = {}

for disease in diseases:
    model.set_adapter(disease)
    messages = [
        {"role": "user", "content": create_content_for_disease_prediction(disease)}
    ]

    # Format the text with ChatML tags and generation prompt
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Convert text to tensor numbers and move to GPU
    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=2,  # Set a limit on the number of new tokens to generate (you can adjust this as needed)
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
    )

    # Strip out the input prompt so we only see the newly generated answer
    generated_ids = output[0][inputs.input_ids.shape[1] :]

    results[disease] = tokenizer.decode(generated_ids, skip_special_tokens=True)

print("FINE-TUNED ANSWERS:\n")

for disease, answer in results.items():
    print(f"{disease}: {answer}")

FINE-TUNED ANSWERS:

diabetes: negative

cancer: negative


In [ ]:
import torch

print(torch.cuda.get_device_name(0))
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

total_vram = torch.cuda.get_device_properties(0).total_memory

print(f"Total VRAM: {total_vram / 1024**3:.2f} GB")
print(f"Allocated:  {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved:   {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

print(torch.cuda.memory_summary())

NVIDIA GeForce RTX 3060 Laptop GPU
Allocated: 4.54 GB
Reserved: 4.68 GB
Total VRAM: 6.00 GB
Allocated:  4.54 GB
Reserved:   4.68 GB
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   4649 MiB |   5457 MiB |  18244 GiB |  18239 GiB |
|       from large pool |   4433 MiB |   5305 MiB |  15524 GiB |  15520 GiB |
|       from small pool |    216 MiB |    286 MiB |   2719 GiB |   2719 GiB |
|---------------------------------------------------------------------------|
| Active m